# Tarea B — Temperatura, top-p y entropía de la distribución de salida

## Parte 1 — Softmax con temperatura

In [1]:
# Subtarea T1: Implementar en NumPy la softmax con temperatura numéricamente estable (restar el máximo de los logits) sobre los logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.
import json
import numpy as np

# ----------------------------------------------------------------------
# SUBTAREA T1: Softmax con temperatura (numéricamente estable) y entropía
# Datos del enunciado: vocabulario de seis tokens t0–t5
# ----------------------------------------------------------------------

tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


def softmax_con_temperatura(z, T):
    """Softmax con temperatura p_i = exp(z_i/T) / sum_j exp(z_j/T).

    Numéricamente estable: se resta el máximo de los logits escalados
    antes de aplicar exp, para evitar desbordamiento.
    """
    z = np.asarray(z, dtype=float) / float(T)
    z = z - np.max(z)  # estabilidad numérica
    e = np.exp(z)
    return e / np.sum(e)


def entropia_bits(p):
    """Entropía de Shannon en bits: H = -sum p_i * log2(p_i)."""
    p = np.asarray(p, dtype=float)
    mascara = p > 0  # evita log2(0); 0*log2(0) se define como 0
    return float(-np.sum(p[mascara] * np.log2(p[mascara])))


# ----------------------------------------------------------------------
# Cálculo de distribuciones y entropías para cada temperatura
# ----------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": [float(x) for x in logits],
    "temperaturas": [float(T) for T in temperaturas],
    "distribuciones": {},
    "entropias_bits": {},
    "suma_distribucion": {},
    "prob_max": {},
}

for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    H = entropia_bits(p)
    clave = f"T={T}"
    resultados["distribuciones"][clave] = {
        tok: float(pi) for tok, pi in zip(tokens, p)
    }
    resultados["entropias_bits"][clave] = float(H)
    resultados["suma_distribucion"][clave] = float(np.sum(p))
    resultados["prob_max"][clave] = float(np.max(p))

# ----------------------------------------------------------------------
# Guardar todas las cifras (sin redondear) en resultados.json
# ----------------------------------------------------------------------
with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

# ----------------------------------------------------------------------
# Tabla con cuatro decimales: distribuciones y entropías
# ----------------------------------------------------------------------
col_w = 10
encabezado = "Token ".ljust(7) + "".join(f"T={T}".rjust(col_w) for T in temperaturas)
separador = "-" * len(encabezado)

print("Softmax con temperatura sobre logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]")
print()
print(encabezado)
print(separador)
for i, tok in enumerate(tokens):
    fila = f"{tok}".ljust(7)
    for T in temperaturas:
        fila += f"{resultados['distribuciones'][f'T={T}'][tok]:.4f}".rjust(col_w)
    print(fila)
print(separador)
fila_suma = "Suma  ".ljust(7)
for T in temperaturas:
    fila_suma += f"{resultados['suma_distribucion'][f'T={T}']:.4f}".rjust(col_w)
print(fila_suma)
fila_H = "H (bits)".ljust(7)
for T in temperaturas:
    fila_H += f"{resultados['entropias_bits'][f'T={T}']:.4f}".rjust(col_w)
print(fila_H)
print(separador)

# Cifras principales con 4 decimales
print()
print("Cifras principales:")
print(f"Logits: {[float(x) for x in logits]}")
for T in temperaturas:
    clave = f"T={T}"
    dist_str = ", ".join(
        f"{tok}={resultados['distribuciones'][clave][tok]:.4f}" for tok in tokens
    )
    print(f"T={T}: p=[{dist_str}]")
    print(
        f"        H={resultados['entropias_bits'][clave]:.4f} bits, "
        f"p_max={resultados['prob_max'][clave]:.4f}, "
        f"suma={resultados['suma_distribucion'][clave]:.4f}"
    )


Softmax con temperatura sobre logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

Token       T=0.5     T=1.0     T=2.0
-------------------------------------
t0         0.8231    0.5516    0.3583
t1         0.1114    0.2029    0.2173
t2         0.0410    0.1231    0.1693
t3         0.0225    0.0912    0.1457
t4         0.0020    0.0275    0.0800
t5         0.0000    0.0037    0.0294
-------------------------------------
Suma       1.0000    1.0000    1.0000
H (bits)    0.9147    1.7998    2.2888
-------------------------------------

Cifras principales:
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]
T=0.5: p=[t0=0.8231, t1=0.1114, t2=0.0410, t3=0.0225, t4=0.0020, t5=0.0000]
        H=0.9147 bits, p_max=0.8231, suma=1.0000
T=1.0: p=[t0=0.5516, t1=0.2029, t2=0.1231, t3=0.0912, t4=0.0275, t5=0.0037]
        H=1.7998 bits, p_max=0.5516, suma=1.0000
T=2.0: p=[t0=0.3583, t1=0.2173, t2=0.1693, t3=0.1457, t4=0.0800, t5=0.0294]
        H=2.2888 bits, p_max=0.3583, suma=1.0000


Implementé la softmax restando el logit máximo antes de exponenciar: así `exp` nunca recibe números grandes y no hay desbordes, y el resultado no cambia porque la softmax es invariante a desplazamientos de los logits. Con los logits dados, el efecto de la temperatura es el esperado: al bajarla la masa se concentra en t0 y al subirla se reparte entre todos. p(t0) pasa de 0.8231 con T=0.5, a 0.5516 con T=1 y a 0.3583 con T=2, mientras que t5, que con T=0.5 casi no tiene masa (0.0000 a cuatro decimales), sube hasta 0.0294 con T=2. La entropía cuenta la misma historia en bits: 0.9147 con T=0.5, 1.7998 con T=1 y 2.2888 con T=2; a mayor temperatura, más incertidumbre. Las tres distribuciones suman 1.0000, como debe ser.

## Parte 2 — Muestreo de núcleo (top-p)

In [2]:
# Subtarea T2: Sobre la distribución de T=1 de la Parte 1, aplicar muestreo de núcleo top-p con p=0.9: ordenar los tokens por probabilidad descendente, conservar el 
import json
import numpy as np

# ============================================================
# SUBTAREA T2: Muestreo de núcleo (top-p) con p = 0.9
# sobre la distribución de T = 1 de la Parte 1 (recalculada)
# ============================================================

# ---------- Datos del enunciado ----------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

# ---------- Softmax con temperatura (numéricamente estable) ----------
def softmax_con_temperatura(z, T):
    z = np.asarray(z, dtype=float)
    escalados = z / float(T)
    escalados = escalados - escalados.max()  # estabilidad numérica
    ex = np.exp(escalados)
    return ex / ex.sum()

T = 1.0
p = softmax_con_temperatura(logits, T)

def entropia_bits(probs):
    probs = np.asarray(probs, dtype=float)
    mask = probs > 0
    return float(-(probs[mask] * np.log2(probs[mask])).sum())

H_original = entropia_bits(p)

# ---------- Top-p (nucleus) con p = 0.9 ----------
p_nucleo = 0.9

# Ordenar tokens por probabilidad descendente (estable: desempates por índice)
orden = np.argsort(-p, kind="stable")
tokens_ordenados = [tokens[i] for i in orden]
probs_ordenadas = p[orden]
logits_ordenados = logits[orden]
acumulada = np.cumsum(probs_ordenadas)

# Conjunto mínimo cuya probabilidad acumulada alcanza p:
# se añaden tokens en orden descendente hasta que la suma acumulada >= p
# (el token que cruza el umbral se incluye). Tolerancia por punto flotante.
alcanza = acumulada >= (p_nucleo - 1e-12)
if alcanza.any():
    k = int(np.argmax(alcanza)) + 1  # tamaño del núcleo
else:
    k = len(tokens)  # salvaguarda (no debería ocurrir: la masa total es 1)

tokens_sobrevivientes = tokens_ordenados[:k]
probs_nucleo = probs_ordenadas[:k]
masa_nucleo = float(acumulada[k - 1])
masa_antes_del_ultimo = float(acumulada[k - 2]) if k >= 2 else 0.0
probs_renormalizadas = probs_nucleo / masa_nucleo

tokens_excluidos = tokens_ordenados[k:]
masa_excluida = float(probs_ordenadas[k:].sum())

# Distribución renormalizada completa (0.0 para los tokens excluidos)
dist_renorm_completa = {t: 0.0 for t in tokens}
for t, pr in zip(tokens_sobrevivientes, probs_renormalizadas):
    dist_renorm_completa[t] = float(pr)

H_renormalizada = entropia_bits(probs_renormalizadas)

# Verificaciones del criterio de conjunto mínimo
verif_alcanza = bool(masa_nucleo >= p_nucleo - 1e-12)
verif_minima = bool(masa_antes_del_ultimo < p_nucleo - 1e-12) if k >= 2 else True

# ---------- Guardar todas las cifras en resultados.json ----------
resultados = {
    "tokens": tokens,
    "logits": [float(z) for z in logits],
    "temperatura": float(T),
    "p_nucleo": float(p_nucleo),
    "distribucion_T1": {t: float(v) for t, v in zip(tokens, p)},
    "entropia_original_bits": H_original,
    "orden_descendente": tokens_ordenados,
    "logits_ordenados": [float(v) for v in logits_ordenados],
    "probs_ordenadas": [float(v) for v in probs_ordenadas],
    "prob_acumulada_ordenada": [float(v) for v in acumulada],
    "tamano_nucleo": int(k),
    "tokens_sobrevivientes": tokens_sobrevivientes,
    "tokens_excluidos": tokens_excluidos,
    "masa_nucleo_antes_renormalizar": masa_nucleo,
    "masa_acumulada_antes_del_ultimo_token": masa_antes_del_ultimo,
    "masa_excluida": masa_excluida,
    "distribucion_renormalizada": {
        t: float(v) for t, v in zip(tokens_sobrevivientes, probs_renormalizadas)
    },
    "distribucion_renormalizada_completa": dist_renorm_completa,
    "suma_renormalizada": float(probs_renormalizadas.sum()),
    "entropia_renormalizada_bits": H_renormalizada,
    "verificacion_alcanza_p": verif_alcanza,
    "verificacion_conjunto_minimo": verif_minima,
}

with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

# ---------- Impresión de las cifras principales (4 decimales) ----------
print("Muestreo de núcleo (top-p) sobre la distribución de T=1")
print(f"Logits: {[float(z) for z in logits]}")
print(f"Temperatura T = {T:.1f}   |   p = {p_nucleo:.1f}")
print()

print("Tokens ordenados por probabilidad descendente:")
print("Token   Prob.      Acumulada   En núcleo")
print("-" * 44)
for t, pr, ac in zip(tokens_ordenados, probs_ordenadas, acumulada):
    dentro = "sí" if t in tokens_sobrevivientes else "no"
    print(f"{t}      {pr:.4f}    {ac:.4f}      {dentro}")
print("-" * 44)

print()
print(f"Conjunto mínimo con masa acumulada >= {p_nucleo:.1f}: "
      f"{tokens_sobrevivientes} (masa = {masa_nucleo:.4f})")
print(f"Tokens excluidos: {tokens_excluidos} "
      f"(masa excluida = {masa_excluida:.4f})")
print()

print("Distribución renormalizada dentro del núcleo:")
print("Token   Original   Renormalizada")
print("-" * 34)
for t, pr, rn in zip(tokens_sobrevivientes, probs_nucleo, probs_renormalizadas):
    print(f"{t}      {pr:.4f}     {rn:.4f}")
print("-" * 34)
print(f"Suma renormalizada: {float(probs_renormalizadas.sum()):.4f}")
print()

print("Cifras principales:")
print(f"Tokens sobrevivientes (top-p, p={p_nucleo:.1f}): {tokens_sobrevivientes}")
print(f"Tamaño del núcleo: {k}")
print(f"Masa del núcleo antes de renormalizar: {masa_nucleo:.4f}")
print("Distribución renormalizada: "
      + ", ".join(f"{t}={v:.4f}" for t, v in zip(tokens_sobrevivientes, probs_renormalizadas)))
print(f"Suma: {float(probs_renormalizadas.sum()):.4f}")
print(f"Entropía original (T=1): {H_original:.4f} bits")
print(f"Entropía renormalizada:  {H_renormalizada:.4f} bits")


Muestreo de núcleo (top-p) sobre la distribución de T=1
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]
Temperatura T = 1.0   |   p = 0.9

Tokens ordenados por probabilidad descendente:
Token   Prob.      Acumulada   En núcleo
--------------------------------------------
t0      0.5516    0.5516      sí
t1      0.2029    0.7546      sí
t2      0.1231    0.8776      sí
t3      0.0912    0.9688      sí
t4      0.0275    0.9963      no
t5      0.0037    1.0000      no
--------------------------------------------

Conjunto mínimo con masa acumulada >= 0.9: ['t0', 't1', 't2', 't3'] (masa = 0.9688)
Tokens excluidos: ['t4', 't5'] (masa excluida = 0.0312)

Distribución renormalizada dentro del núcleo:
Token   Original   Renormalizada
----------------------------------
t0      0.5516     0.5694
t1      0.2029     0.2095
t2      0.1231     0.1270
t3      0.0912     0.0941
----------------------------------
Suma renormalizada: 1.0000

Cifras principales:
Tokens sobrevivientes (top-p, p=0.9): ['t0', 't1'

Ordené los tokens de mayor a menor probabilidad (con T=1 el orden coincide con el de los logits: t0, t1, t2, t3, t4, t5) y fui acumulando. Tras t2 la suma es 0.8776, que todavía no alcanza 0.9; al incluir t3 llega a 0.9688. Por eso el núcleo mínimo es {t0, t1, t2, t3}, y t4 y t5 quedan fuera con una masa excluida de 0.0312. Para renormalizar dividí cada probabilidad del núcleo entre 0.9688, y quedaron t0=0.5694, t1=0.2095, t2=0.1270 y t3=0.0941, que suman 1.0000. Un detalle que me pareció interesante: la entropía baja de 1.7998 a 1.6341 bits, porque recortar la cola reduce las opciones posibles y con ello la incertidumbre del muestreo.

## Parte 3 — Comprobación empírica

In [3]:
# Subtarea T3: Tomar 10000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Comparar frecuencias observadas vs probabilidades
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Datos del enunciado (Tarea B)
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

T = 1.0           # temperatura de la Parte 1 usada en la Parte 2
p_nucleo = 0.9    # umbral top-p de la Parte 2
n_muestras = 10000
semilla = 0

# ----------------------------------------------------------------------
# Paso 1: softmax con temperatura (numéricamente estable) -> distribución T=1
# ----------------------------------------------------------------------
z = logits / T
z = z - np.max(z)                 # estabilidad numérica
exp_z = np.exp(z)
probs_T1 = exp_z / np.sum(exp_z)

# ----------------------------------------------------------------------
# Paso 2: muestreo de núcleo (top-p, p=0.9) y renormalización (Parte 2)
# ----------------------------------------------------------------------
orden = np.argsort(-probs_T1, kind="stable")     # orden descendente
probs_ordenadas = probs_T1[orden]
acumulada = np.cumsum(probs_ordenadas)

# conjunto mínimo cuya probabilidad acumulada alcanza p
k = int(np.searchsorted(acumulada, p_nucleo, side="left")) + 1
idx_nucleo = orden[:k]
masa_nucleo = float(probs_ordenadas[:k].sum())

p_renorm_nucleo = probs_ordenadas[:k] / masa_nucleo

# distribución renormalizada sobre el vocabulario completo (ceros fuera del núcleo)
p_teorica = np.zeros(len(tokens), dtype=float)
p_teorica[idx_nucleo] = p_renorm_nucleo
p_teorica = p_teorica / p_teorica.sum()   # normalización exacta para rng.choice

# entropía teórica de la distribución renormalizada (bits)
mask_teor = p_teorica > 0
entropia_teorica = float(-np.sum(p_teorica[mask_teor] * np.log2(p_teorica[mask_teor])))

# ----------------------------------------------------------------------
# Paso 3: 10000 muestras de la distribución renormalizada con default_rng(0)
# ----------------------------------------------------------------------
rng = np.random.default_rng(semilla)
muestras = rng.choice(len(tokens), size=n_muestras, p=p_teorica)

conteos = np.bincount(muestras, minlength=len(tokens))
frecuencias = conteos / n_muestras

# ----------------------------------------------------------------------
# Paso 4: divergencia KL (bits) de la empírica respecto de la teórica
#   KL(P_emp || P_teor) = sum_i p_emp_i * log2(p_emp_i / p_teor_i)
#   (términos con p_emp_i = 0 contribuyen 0 por convención)
# ----------------------------------------------------------------------
mask_emp = frecuencias > 0
kl_bits = float(
    np.sum(frecuencias[mask_emp] * np.log2(frecuencias[mask_emp] / p_teorica[mask_emp]))
)

entropia_empirica = float(-np.sum(frecuencias[mask_emp] * np.log2(frecuencias[mask_emp])))

errores_abs = np.abs(frecuencias - p_teorica)
error_max = float(errores_abs.max())
error_medio = float(errores_abs.mean())

# ----------------------------------------------------------------------
# Figura: barras agrupadas, frecuencias observadas vs probabilidades teóricas
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(tokens))
ancho = 0.38

b1 = ax.bar(x - ancho / 2, frecuencias, ancho,
            label="Frecuencia observada (empírica)",
            color="#4C72B0", edgecolor="black", linewidth=0.5)
b2 = ax.bar(x + ancho / 2, p_teorica, ancho,
            label="Probabilidad teórica (top-p renormalizada)",
            color="#DD8452", edgecolor="black", linewidth=0.5)

for barras in (b1, b2):
    for b in barras:
        h = b.get_height()
        ax.annotate(f"{h:.4f}",
                    xy=(b.get_x() + b.get_width() / 2, h),
                    xytext=(0, 2), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad")
ax.set_ylim(0, max(frecuencias.max(), p_teorica.max()) * 1.18)
ax.set_title(f"Parte 3 - Comprobación empírica del muestreo top-p\n"
             f"n = {n_muestras}, semilla = {semilla}, p = {p_nucleo}   |   "
             f"KL(empírica || teórica) = {kl_bits:.4f} bits")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("t3_frecuencias_vs_teoricas.png", dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------
# Resultados -> resultados.json
# ----------------------------------------------------------------------
resultados = {
    "n_muestras": int(n_muestras),
    "semilla": int(semilla),
    "tokens": tokens,
    "logits": [float(v) for v in logits],
    "temperatura": float(T),
    "p_nucleo": float(p_nucleo),
    "tokens_sobrevivientes": [tokens[i] for i in idx_nucleo],
    "tamano_nucleo": int(k),
    "masa_nucleo_antes_renormalizar": masa_nucleo,
    "distribucion_renormalizada_teorica": {t: float(p) for t, p in zip(tokens, p_teorica)},
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(f) for t, f in zip(tokens, frecuencias)},
    "error_absoluto_por_token": {t: float(e) for t, e in zip(tokens, errores_abs)},
    "error_absoluto_maximo": error_max,
    "error_absoluto_medio": error_medio,
    "suma_frecuencias_observadas": float(frecuencias.sum()),
    "entropia_teorica_bits": entropia_teorica,
    "entropia_empirica_bits": entropia_empirica,
    "kl_empirica_respecto_teorica_bits": kl_bits,
    "figura": "t3_frecuencias_vs_teoricas.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# Salida por consola (4 decimales)
# ----------------------------------------------------------------------
print("Parte 3 - Comprobación empírica del muestreo top-p")
print(f"n = {n_muestras} muestras, numpy.random.default_rng({semilla}), p = {p_nucleo}")
print(f"Tokens sobrevivientes (núcleo): {[tokens[i] for i in idx_nucleo]}")
print()
print("Token   Teórica   Observada   Conteo   |Error abs.|")
print("----------------------------------------------------")
for i, t in enumerate(tokens):
    print(f"{t:5s}   {p_teorica[i]:.4f}    {frecuencias[i]:.4f}    {conteos[i]:5d}     {errores_abs[i]:.4f}")
print("----------------------------------------------------")
print(f"Suma frecuencias observadas: {frecuencias.sum():.4f}")
print()
print("Cifras principales:")
print(f"Divergencia KL(empírica || teórica) = {kl_bits:.4f} bits")
print(f"Entropía teórica (renormalizada):  {entropia_teorica:.4f} bits")
print(f"Entropía empírica:                 {entropia_empirica:.4f} bits")
print(f"Error absoluto máximo por token:   {error_max:.4f}")
print(f"Error absoluto medio:              {error_medio:.4f}")
print("Figura guardada: t3_frecuencias_vs_teoricas.png")
print("Resultados guardados en resultados.json")


Parte 3 - Comprobación empírica del muestreo top-p
n = 10000 muestras, numpy.random.default_rng(0), p = 0.9
Tokens sobrevivientes (núcleo): ['t0', 't1', 't2', 't3']

Token   Teórica   Observada   Conteo   |Error abs.|
----------------------------------------------------
t0      0.5694    0.5682     5682     0.0012
t1      0.2095    0.2093     2093     0.0002
t2      0.1270    0.1296     1296     0.0026
t3      0.0941    0.0929      929     0.0012
t4      0.0000    0.0000        0     0.0000
t5      0.0000    0.0000        0     0.0000
----------------------------------------------------
Suma frecuencias observadas: 1.0000

Cifras principales:
Divergencia KL(empírica || teórica) = 0.0001 bits
Entropía teórica (renormalizada):  1.6341 bits
Entropía empírica:                 1.6362 bits
Error absoluto máximo por token:   0.0026
Error absoluto medio:              0.0009
Figura guardada: t3_frecuencias_vs_teoricas.png
Resultados guardados en resultados.json


Muestreé 10000 veces la distribución renormalizada con `numpy.random.default_rng(0)` y comparé frecuencias observadas contra probabilidades teóricas. Los valores quedaron muy cerca: t0=0.5682 observado frente a 0.5694 teórico, t1=0.2093 frente a 0.2095, t2=0.1296 frente a 0.1270 y t3=0.0929 frente a 0.0941; t4 y t5 no aparecieron ni una sola vez, como corresponde porque tienen probabilidad 0. El error absoluto máximo fue 0.0026 (en t2) y la divergencia KL de la distribución empírica respecto de la teórica dio 0.0001 bits, prácticamente cero: la diferencia es solo el ruido estadístico esperable al muestrear. En el gráfico de barras cada pareja de barras (teórica y observada) es casi indistinguible, que es justo lo que quería comprobar.

![Frecuencias observadas frente a probabilidades teóricas tras 10000 muestreos del núcleo top-p](T3_t3_frecuencias_vs_teoricas.png)

## Parte 4 — Pregunta conceptual

Cuando T → 0⁺, el cociente z_i/T se hace muy grande en magnitud y las diferencias entre logits se amplifican exponencialmente: el token de mayor logit (t0) se lleva prácticamente toda la masa y los demás tienden a 0, de modo que la distribución se convierte en una masa puntual sobre el argmax. La evidencia están en las entropías de la Parte 1: al bajar T de 2 a 1 y de 1 a 0.5, la entropía cae de 2.2888 a 1.7998 y luego a 0.9147 bits; la tendencia apunta a 0 bits en el límite, que es la entropía de una distribución sin incertidumbre. En la misma dirección, p(t0) crece de 0.3583 a 0.5516 y a 0.8231.

Cuando T → ∞ ocurre lo contrario: z_i/T → 0 para todos los tokens, todos los exp(z_i/T) valen lo mismo y la distribución tiende a la uniforme, con la misma probabilidad para los seis tokens (1/6 cada uno). Las entropías también lo muestran: suben con la temperatura (0.9147 → 1.7998 → 2.2888 bits) acercándose al máximo posible para seis tokens, y las probabilidades se emparejan: t0 baja hasta 0.3583 y hasta t5, el peor token, llega a 0.0294.

Que T → 0 equivalga a la decodificación voraz se ve así: greedy elige en cada paso el token de mayor logit, es decir, el argmax. En el límite T → 0 la softmax pone toda la probabilidad en ese mismo argmax, así que muestrear de esa distribución devuelve el argmax con probabilidad 1: el muestreo se vuelve determinista y coincide con greedy. Dicho de otra forma, con entropía 0 no queda aleatoriedad que explotar y "muestrear" y "tomar el máximo" son la misma operación. Por eso, en la práctica, una temperatura muy baja se usa como una versión aproximada de la decodificación voraz.